In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [17]:
# Load the Dataset
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [18]:
## Preprocess the data
# Drop irrelevant columns
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
# Encode categorical varaibles
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])# One Hot encoding : "Geography"
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo = OneHotEncoder()
geo_encoder = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [19]:
import numpy as np
geo_column  = np.array(onehot_encoder_geo.get_feature_names_out())

In [20]:
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=geo_column)

In [21]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [22]:
# Combine OHE with original data
data = pd.concat([data.drop('Geography',axis=1), geo_encoded_df],axis=1)

In [23]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [24]:
# Split the data into features and target
X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']

In [25]:
## Split the data in training and testing sets
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

# First, train-test split then STANDARDIZATION is done in order to PREVENT DATA-LEAKAGE COMPLETELY.

In [26]:
## Scale these features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [27]:
# Save the encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

# ANN Regression Problem Statement

In [28]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [31]:
# Build the model
model = Sequential([
    Dense(64, activation='relu', input_shape = (X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1) # Output layer for regression where linear activation function is used by-default.
])

# Compile the Model
model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])

In [32]:
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 64)                832       
                                                                 
 dense_7 (Dense)             (None, 32)                2080      
                                                                 
 dense_8 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [33]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

# Set up TensorBoard
log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [ ]:
# Set up Early Stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [35]:
# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[early_stopping_callback, tensorboard_callback]
)

Epoch 1/100


250/250 [==============================] - 4s 5ms/step - loss: 100386.2344 - mae: 100386.2344 - val_loss: 98552.3359 - val_mae: 98552.3359
Epoch 2/100
250/250 [==============================] - 1s 4ms/step - loss: 99778.0000 - mae: 99778.0000 - val_loss: 97316.8438 - val_mae: 97316.8438
Epoch 3/100
250/250 [==============================] - 1s 4ms/step - loss: 97651.8203 - mae: 97651.8203 - val_loss: 94175.8750 - val_mae: 94175.8750
Epoch 4/100
250/250 [==============================] - 1s 4ms/step - loss: 93410.8750 - mae: 93410.8750 - val_loss: 88781.2266 - val_mae: 88781.2266
Epoch 5/100
250/250 [==============================] - 1s 4ms/step - loss: 87033.1094 - mae: 87033.1094 - val_loss: 81570.7266 - val_mae: 81570.7266
Epoch 6/100
250/250 [==============================] - 1s 4ms/step - loss: 79113.0547 - mae: 79113.0547 - val_loss: 73487.6719 - val_mae: 73487.6719
Epoch 7/100
250/250 [==============================] - 1s 5ms/step - loss: 70853.6875 - mae: 70853.687

In [36]:
%load_ext tensorboard

In [38]:
%tensorboard --logdir regressionlogs/fit/

Reusing TensorBoard on port 6009 (pid 17652), started 0:02:02 ago. (Use '!kill 17652' to kill it.)

In [44]:
# Evaluate model on the test data
test_loss, test_mae = model.evaluate(X_test,y_test)

63/63 [==============================] - 0s 4ms/step - loss: 50271.0000 - mae: 50271.0000


In [45]:
test_loss, test_mae

(50271.0, 50271.0)

In [46]:
model.save('regression_model.h5')

c:\Users\Omkar Raut\Desktop\DataSceince\DL-venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
